# 01 — Data scraper

Ports `data_scraper.py` (Google Images scraper via `undetected_chromedriver`)
to notebook cells. No GPU involved here — this notebook only collects source
photos of each snack to later cut out and composite. Split into setup /
collect / download stages with a small intermediary test before a full pull.

Uses your own already-installed Chromium/Chrome; make sure one is on `PATH`
(Arch: `pacman -S chromium` or `google-chrome`).


In [ ]:
import os
import time
import requests
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By


## Config

In [ ]:
SEARCH_QUERY = "nutri grain strawberry bar"   # change per snack type
MAX_IMAGES = 100


## Driver setup

In [ ]:
def make_driver():
    options = uc.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    return uc.Chrome(options=options, use_automation_extension=False)


## Collection — scroll Google Images and resolve high-res source URLs

In [ ]:
def collect_image_urls(driver, search_query, max_images):
    search_url = f"https://www.google.com/search?q={search_query}&udm=2"
    driver.get(search_url)
    time.sleep(2)

    image_urls = set()
    thumbnails_processed = 0

    print(f"Gathering images for '{search_query}'...")

    while len(image_urls) < max_images:
        thumbnails = driver.find_elements(By.CSS_SELECTOR, "div[data-ri] img, [role='listitem'] img, img")

        if not thumbnails or thumbnails_processed >= len(thumbnails):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            try:
                show_more = driver.find_element(By.CSS_SELECTOR, "input[type='button'], [role='button']")
                if show_more.is_displayed():
                    show_more.click()
                    time.sleep(1)
            except Exception:
                pass

            new_thumbnails = driver.find_elements(By.CSS_SELECTOR, "div[data-ri] img, [role='listitem'] img, img")
            if len(new_thumbnails) == len(thumbnails):
                print("No more thumbnails available on the page.")
                break
            continue

        img = thumbnails[thumbnails_processed]
        thumbnails_processed += 1

        try:
            if img.size["width"] < 50 or img.size["height"] < 50:
                continue

            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", img)
            img.click()
            time.sleep(1.0)

            parent_elements = driver.find_elements(By.CSS_SELECTOR, "a[href*='imgurl'], div[role='dialog'] a")
            for element in parent_elements:
                href = element.get_attribute("href")
                if href and "imgurl=" in href:
                    raw_url = href.split("imgurl=")[1].split("&")[0]
                    clean_url = requests.utils.unquote(raw_url)
                    if clean_url and clean_url.startswith("http") and clean_url not in image_urls:
                        image_urls.add(clean_url)
                        print(f"Found high-res source link {len(image_urls)}: {clean_url[:60]}...")
                        break
        except Exception:
            continue

    return image_urls


## Download phase

In [ ]:
def download_images(image_urls, folder_name):
    os.makedirs(folder_name, exist_ok=True)
    saved = []
    for i, url in enumerate(image_urls):
        try:
            headers = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36"}
            response = requests.get(url, headers=headers, timeout=8)
            if response.status_code == 200:
                ext = "jpg"
                ctype = response.headers.get("Content-Type", "")
                if "image/png" in ctype:
                    ext = "png"
                elif "image/webp" in ctype:
                    ext = "webp"
                file_path = os.path.join(folder_name, f"image_{i + 1}.{ext}")
                with open(file_path, "wb") as f:
                    f.write(response.content)
                saved.append(file_path)
        except Exception:
            continue
    return saved


def download_high_res_images_fallback(search_query, max_images=100):
    folder_name = search_query.replace(" ", "_").lower()
    driver = make_driver()
    try:
        image_urls = collect_image_urls(driver, search_query, max_images)
    finally:
        driver.quit()

    if not image_urls:
        print("Could not resolve any high-res source URLs.")
        return []

    print(f"Downloading {len(image_urls)} high-resolution source files...")
    saved = download_images(image_urls, folder_name)
    print(f"Done! {len(saved)} images saved to /{folder_name}")
    return saved


## Intermediary test — pull 5 images and eyeball them

Always run this before the full-size pull below: confirms the selectors still
match Google's current DOM and that the downloaded files are real photos, not
broken redirects.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

test_files = download_high_res_images_fallback(search_query=SEARCH_QUERY, max_images=5)

fig, axes = plt.subplots(1, max(len(test_files), 1), figsize=(15, 4))
if len(test_files) == 1:
    axes = [axes]
for ax, path in zip(axes, test_files):
    try:
        ax.imshow(Image.open(path))
        ax.set_title(os.path.basename(path), fontsize=8)
    except Exception as e:
        ax.set_title(f"failed: {e}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Full run

Only run once the 5-image test above looks right.

In [ ]:
RUN_FULL_SCRAPE = False  # flip to True to actually run the full pull

if RUN_FULL_SCRAPE:
    download_high_res_images_fallback(search_query=SEARCH_QUERY, max_images=MAX_IMAGES)
